In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
path = '../../dataset_imputed.csv'
df_raw = pd.read_csv(path, header=0)

In [3]:
nulls = df_raw.isnull().sum()
null_df = pd.DataFrame({
    'Nulos'      : nulls,
    '% del total': (nulls / len(df_raw) * 100).round(2)
}).query('Nulos > 0').sort_values('% del total', ascending=False)

print(f"Columnas con nulos: {len(null_df)}")
display(null_df)

Columnas con nulos: 19


,Nulos,% del total
socioeconomic.level,66446,85.72
social.lag,63984,82.54
zone.type,54718,70.59
average.first.period,53010,68.38
failed.subject.first.period,53010,68.38
dropped.subject.first.period,53010,68.38
life.work.mentoring,41824,53.95
student.society.leadership,41824,53.95
art.culture,41824,53.95
athletic.sports,41824,53.95


El dataset tiene varios estudiantes que no estan en undergraduate level. Se quitan los que no estan.

In [4]:
df = df_raw[df_raw['level'] == 'Undergraduate'].copy()
print(f"Universitarios: {len(df):,} / {len(df_raw):,} ({len(df)/len(df_raw)*100:.1f}%)")

Universitarios: 77,517 / 77,517 (100.0%)


# K-Means
Aqui se hace la depuracion para entrenar modelos K-Means

In [5]:
# ── 1.2  Eliminar columnas con fuga o irrelevantes ─────────────────────────
DROP_COLS = [
    'student.id',
    'level',                            # constante tras filtrar
    'average.first.period',             # solo en Tec21
    'failed.subject.first.period',      # solo en Tec21
    'dropped.subject.first.period',     # solo en Tec21
    'dropout.semester',                 # derivado de retention → fuga
    'scholarship.type',                 # redundante con total.scholarship.loan
    'scholarship.perc',                 # redundante porque se tiene total.scholarship.loan
    'program',
    'id.school.origin',                 # se elimina 
    'school.cost',                      # redundante con socioeconomic.level
    'general.math.eval',                # escala inconsistente, baja cobertura. 50% no lo tienen
    'father.exatec', 'mother.exatec',   # redundantes
    'father.education.complete', 'father.education.summary',
    'mother.education.complete', 'mother.education.summary',
    'scholarship.perc', 'loan.perc',    # ya están en total.scholarship.loan
]
df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)

'parents.exatec' se cambia a 0 o 1: donde 1 tiene papas exatec. 0 si tiene 'No' o 'No information'.

In [6]:
df['parents_exatec_enc'] = df['parents.exatec'].map({
    'Yes': 1,
    'No': 0,
    'No information': 0
})
df.drop(columns=['parents.exatec'], inplace=True)

##### M-CAR Missing Completely at Random
Se normaliza los tests tomados antes de ingresar a la universidad. Se normaliza a la escala 0-1. Luego, se imputa la mediana a las columnas admission_test y admission.rubric que tienen Nan.

In [7]:
# MCAR: normalizar admission.test (PAA 400-1600 y PAL 0-100)
def norm_admission_test(val):
    """Normaliza PAA (>100) y PAL (0-100) a escala 0-1. 'Does not apply' → NaN."""
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if s.lower().startswith('does not'):
        return np.nan
    try:
        f = float(s)
        if f > 100:                    # PAA (rango 400–1600)
            return max(0.0, (f - 400) / 1200.0)
        elif f >= 0:                   # PAL (rango 0–100)
            return f / 100.0
    except Exception:
        pass
    return np.nan

df['admission_test_norm'] = df['admission.test'].apply(norm_admission_test)
df.drop(columns=['admission.test'], inplace=True)

# Imputar MCAR con mediana (el test_norm y la rubrica dada)
med_at = df['admission_test_norm'].median()
med_ar = df['admission.rubric'].median()
df['admission_test_norm'] = df['admission_test_norm'].fillna(med_at)
df['admission.rubric'] = df['admission.rubric'].fillna(med_ar)

print(f"admission_test_norm  — median imputado: {med_at:.3f}  | nulos restantes: {df['admission_test_norm'].isna().sum()}")
print(f"admission.rubric     — median imputado: {med_ar:.2f} | nulos restantes: {df['admission.rubric'].isna().sum()}")

admission_test_norm  — median imputado: 0.808  | nulos restantes: 0
admission.rubric     — median imputado: 36.00 | nulos restantes: 0


##### MAR Missing At Random: los datos faltantes tienen un patrón, pero ese patrón depende de otras variables que sí puedes observar.
AD14-AD17: tenían 3 columnas (physical.education, cultural.diffusion, student.society)
AD18-AD20: cambiaron a 5 columnas LiFE (athletic.sports, art.culture, student.society.leadership, life.work.mentoring, wellness.activities) con total.life.activities como contador.

Se crean tres variables binarias que indican si el alumno participó en alguna actividad de cada categoría: 
- física (has_physical) si tiene physical.education, athletic.sports, o wellness.activities. 
- cultural (has_cultural) si tiene cultural.diffusion, o art.culture.
- social (has_society) si tiene student.society, student.society.leadership, o life.work.mentoring. 

Esto reemplaza las 9 columnas originales de actividades, que eran inconsistentes entre generaciones por el cambio de modelo educativo de PreTec21 a Tec21.

In [8]:
ACTIVITY_COLS = [
    'physical.education', 'cultural.diffusion', 'student.society',
    'total.life.activities', 'athletic.sports', 'art.culture',
    'student.society.leadership', 'life.work.mentoring', 'wellness.activities'
]

def parse_activity(val):
    if pd.isna(val):
        return 0
    s = str(val).strip()
    if s in ('Does not apply', 'No information', '', '0', '0.0'):
        return 0
    try:
        return int(float(s) > 0)
    except Exception:
        return 0

# Physical: physical.education, wellness.activities, athletic.sports
df['has_physical'] = df[['physical.education', 'wellness.activities', 'athletic.sports']].apply(
    lambda row: int(any(parse_activity(v) == 1 for v in row)), axis=1
)

# Cultural: cultural.diffusion, art.culture
df['has_cultural'] = df[['cultural.diffusion', 'art.culture']].apply(
    lambda row: int(any(parse_activity(v) == 1 for v in row)), axis=1
)

# Society: student.society, student.society.leadership, life.work.mentoring
df['has_society'] = df[['student.society', 'student.society.leadership', 'life.work.mentoring']].apply(
    lambda row: int(any(parse_activity(v) == 1 for v in row)), axis=1
)

df.drop(columns=[c for c in ACTIVITY_COLS if c in df.columns], inplace=True)

print(f"has_physical  → {df['has_physical'].sum():,} ({df['has_physical'].mean()*100:.1f}%)")
print(f"has_cultural  → {df['has_cultural'].sum():,} ({df['has_cultural'].mean()*100:.1f}%)")
print(f"has_society   → {df['has_society'].sum():,} ({df['has_society'].mean()*100:.1f}%)")

has_physical  → 49,222 (63.5%)
has_cultural  → 30,095 (38.8%)
has_society   → 33,406 (43.1%)


##### MNAR - Missing Not At Random: los datos faltantes no son aleatorios, sino que la ausencia del dato en sí misma tiene significado.
Aqui, la columna 'first.generation' tiene tres opciones:
1. Yes: Donde se convierte a '1'
2. No: Donde se se convierte a '0'
3. No information/Does not apply: Se convierte a '-1'

In [9]:
# first.generation → 3 categorías numéricas
def encode_first_gen(val):
    """Yes=1, No=0, No information/Does not apply=-1 (patrón MNAR; el modelo aprende la ausencia)."""
    s = str(val).strip()
    if s == 'Yes':           return  1
    if s == 'No':            return  0
    return -1   # 'No information' or 'Does not apply'

df['first_gen_enc'] = df['first.generation'].apply(encode_first_gen)
df.drop(columns=['first.generation'], inplace=True)

vals = df['first_gen_enc'].value_counts().sort_index()
print("first_gen_enc distribución:")
print(f"  -1 (No info / MNAR): {vals.get(-1,0):>6,} ({vals.get(-1,0)/len(df)*100:.1f}%)")
print(f"   0 (No)            : {vals.get(0,0):>6,} ({vals.get(0,0)/len(df)*100:.1f}%)")
print(f"   1 (Yes)           : {vals.get(1,0):>6,} ({vals.get(1,0)/len(df)*100:.1f}%)")

first_gen_enc distribución:
  -1 (No info / MNAR): 37,372 (48.2%)
   0 (No)            : 34,752 (44.8%)
   1 (Yes)           :  5,393 (7.0%)


Se genera un numero sobre el degree que tienen los papas

In [10]:
EDU_ORD = {
    'No information':       -1,
    'No degree':             0,
    'Undergraduate degree':  1,
    'Master degree':         2,
    'PhD':                   3,
}

df['educ_padres_max'] = df['max.degree.parents'].map(EDU_ORD).astype(int)
df.drop(columns=['max.degree.parents'], inplace=True)


Se cambia columna de genero a una columna si es hombre o no.

In [11]:
df['is_male'] = df['gender'].map({'Male': 1, 'Female': 0}).astype(int)
df.drop(columns=['gender'], inplace=True)

Se cambia tec.no.tec a si ha estado en prepa tec o no.

In [12]:
df['estuvo.prepa_tec'] = df['tec.no.tec'].map({'TEC': 1, 'NO TEC': 0}).astype(int)
df.drop(columns=['tec.no.tec'], inplace=True)

** Duda ** Si queremos crear columnas para saber si es foreigner o no para kmeans?


 Se reemplaza la columna foreign por dos columnas binarias (foreign_Yes: National y foreign_Yes: Foreigner), donde cada una vale 1 si el alumno pertenece a esa categoría y 0 si no es foreigner.

In [13]:
# foreign_dummies = pd.get_dummies(df['foreign'], prefix='foreign', drop_first=True, dtype=int)
# df = pd.concat([df, foreign_dummies], axis=1)
df.drop(columns=['foreign'], inplace=True)

** Duda ** Queremos crear nuevas columnas para las zonas en las que esta la persona?

Crea nuevas columnas para saber si esta en urban, rural, semiurban. Las filas con 'No information' se convierten a Nan para que al momento de aplicar One-Hot encoding.

In [14]:
# Primero reemplazar 'No information' con NaN para que get_dummies lo ignore
# df['zone.type'] = df['zone.type'].replace('No information', None)

# zone_dummies = pd.get_dummies(df['zone.type'], prefix='zone', drop_first=False, dtype=int)
# df = pd.concat([df, zone_dummies], axis=1)
df.drop(columns=['zone.type'], inplace=True)

Se cambian a numeros los niveles socio-economicos.

In [15]:
# socioeconomic.level → ordinal 0-7
sec_map = {'No information': 0, 'Level 1': 1, 'Level 2': 2, 'Level 3': 3,
           'Level 4': 4, 'Level 5': 5, 'Level 6': 6, 'Level 7': 7}
df['socioec_enc'] = df['socioeconomic.level'].map(sec_map).fillna(0).astype(int)
df.drop(columns=['socioeconomic.level'], inplace=True)

# social.lag
lag_map = {'No information': 0, 'Low': 1, 'Medium': 2, 'High': 3}
df['social_lag_enc'] = df['social.lag'].map(lag_map).fillna(0).astype(int)
df.drop(columns=['social.lag'], inplace=True)

** Dudas** Deberiamos de quitar region y school? Porque se estan haciendo muchas nuevas columnas, y creo que se estan haciendo muchos datos. 


Crea un one hot encoding para cada escuela del Tec. Se crea cada columna por escuela (i.e. Ingenieria, Negocios, etc.). Se hace lo mismo para la region

In [16]:
# region
# region_dummies = pd.get_dummies(df['region'], prefix='region', drop_first=False, dtype=int)
# df = pd.concat([df, region_dummies], axis=1)
df.drop(columns=['region'], inplace=True)

# school
# school_dummies = pd.get_dummies(df['school'], prefix='school', drop_first=False, dtype=int)
# df = pd.concat([df, school_dummies], axis=1)
df.drop(columns=['school'], inplace=True)

#### Creacion de archivo CSV
Se guardan estos datos en un csv

In [17]:
df.to_csv('../dataset_k_means.csv', index=False)
print("Guardado en ../dataset_k_means.csv")

Guardado en ../dataset_k_means.csv
